<h2>Exercise 04. A/B testing</h2>

In [1]:
import pandas as pd 
import sqlite3

### 1. Use the sqlite3 library to create a connection to the database.

In [2]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

### 2. Using one query per group, create two dataframes, test_results and control_results, with the columns "time" and "avg_diff" and two rows.

In [27]:
test_2345 = pd.io.sql.read_sql(
    '''
    SELECT * FROM control
    
    LIMIT 5
    ''',
    conn
)

test_2345

,uid,labname,first_commit_ts,first_view_ts
0,user_4,project1,2020-04-17 05:19:02.744528,1970-01-19 09:05:48.005761
1,user_4,laba04,2020-04-17 11:33:17.366400,1970-01-19 09:05:48.005761
2,user_4,laba04s,2020-04-17 11:48:41.992466,1970-01-19 09:05:48.005761
3,user_2,laba04,2020-04-18 13:42:35.482008,1970-01-19 09:05:48.005761
4,user_2,laba04s,2020-04-18 13:51:22.291271,1970-01-19 09:05:48.005761


In [7]:
pd.read_sql('SELECT name FROM sqlite_master WHERE type="table";', conn)


,name
0,pageviews
1,checker
2,deadlines
3,datamart
4,test
5,control


In [28]:
test_results = pd.io.sql.read_sql(
    '''
    SELECT 
    CASE 
        WHEN t.first_commit_ts > t.first_view_ts THEN 'after'
        WHEN t.first_commit_ts < t.first_view_ts THEN 'before'
    END as time,
    AVG((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600.0) as avg_diff
    FROM test AS t
    LEFT JOIN deadlines AS d ON t.labname = d.labs
    WHERE t.labname != "project1"
    GROUP BY time
    ''',
    conn
)

test_results

,time,avg_diff
0,after,-103.953446
1,before,-61.156632


In [ ]:
control_results = pd.io.sql.read_sql(
    '''
    SELECT 
    CASE 
        WHEN t.first_commit_ts > t.first_view_ts THEN 'after'
        WHEN t.first_commit_ts < t.first_view_ts THEN 'before'
    END as time,
    AVG((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600.0) as avg_diff
    FROM control AS t
    LEFT JOIN deadlines AS d ON t.labname = d.labs
    WHERE t.labname != "project1"
    GROUP BY time
    ''',
    conn
)

control_results

,time,avg_diff
0,after,-113.232346
1,before,-99.901448


In [31]:
conn.close()

<h2>Гипотеза подтвердилась.


Страница влияет на поведение студентов — после первого посещения они начинают сдавать задания раньше.</h2>